In [ ]:
import pandas as pd
model_df=pd.read_csv("../data/processed/leishmania_ml_dataset.csv")
model_df.head()

In [ ]:
import rdkit
print(rdkit.__version__)

## Generation of Molecular Objects

SMILES strings were converted into RDKit molecular objects to enable the calculation of molecular fingerprints and other structure-based representations required for machine learning.

In [ ]:
from rdkit import Chem

model_df["Mol"] = model_df["SMILES"].apply(
    Chem.MolFromSmiles
)

model_df["Mol"].head()

In [ ]:
model_df["Mol"].isnull().sum()

## Generation of Morgan Fingerprints

Morgan fingerprints were generated from molecular structures to capture structural patterns and chemical substructures. These fingerprints provide a high-dimensional representation of molecular features commonly used in QSAR modeling.

In [ ]:
from rdkit.Chem import AllChem

fp = AllChem.GetMorganFingerprintAsBitVect(
    model_df["Mol"].iloc[0],
    radius=2,
    nBits=1024
)

len(fp)

In [ ]:
import numpy as np

fingerprints = []

for mol in model_df["Mol"]:
    
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=1024
    )
    
    fingerprints.append(
        np.array(fp)
    )

In [ ]:
fp_df = pd.DataFrame(
    fingerprints
)

fp_df.shape

## Construction of Feature Sets

Three feature representations were prepared for model development: molecular descriptors alone, Morgan fingerprints alone, and a combined descriptor-fingerprint representation. This approach allows comparison of different molecular representations for predicting anti-leishmanial activity.

In [ ]:
descriptor_cols = [
    "Molecular_Weight",
    "AlogP",
    "TPSA",
    "HBA",
    "HBD",
    "RO5_Violations",
    "Rotatable_Bonds",
    "QED",
    "Aromatic_Rings",
    "Heavy_Atoms",
    "NP_Likeness"
]

X_desc = model_df[
    descriptor_cols
]

y = model_df["pIC50"]

print(X_desc.shape)

In [ ]:
X_fp = fp_df.copy()

print(X_fp.shape)

In [ ]:
X_combined = pd.concat(
    [
        X_desc.reset_index(drop=True),
        X_fp.reset_index(drop=True)
    ],
    axis=1
)

print(X_combined.shape)

In [ ]:
X_desc.to_csv(
    "../data/processed/X_descriptors.csv",
    index=False
)
X_fp.to_csv(
    "../data/processed/X_fingerprints.csv",
    index=False
)
X_combined.to_csv(
    "../data/processed/X_combined.csv",
    index=False
)
y.to_csv(
    "../data/processed/y_pIC50.csv",
    index=False
)

## Export of Feature Matrices

The descriptor, fingerprint, and combined feature matrices were exported for subsequent machine learning analyses. The target variable (pIC50) was also saved separately to facilitate reproducible model development and evaluation.